# Global Corporate Counterparty Rating (GCCR)
## Notebook 01: SEC EDGAR Ingestion, Preprocessing & Fundamental Feature Engineering

In this notebook, we demonstrate wholesale credit fundamentals ingestion, sector-specific missing value imputation, outlier winsorization, and institutional solvency/liquidity ratio engineering.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src modules are resolvable
sys.path.append("..")

from src.data.edgar_loader import load_sec_financials, load_corporate_defaults, merge_financials_with_defaults
from src.data.preprocessor import preprocess_financial_data
from src.features.financial_ratios import compute_financial_ratios

print("Setup completed successfully.")


### 1. Ingest SEC EDGAR Filings & Merge Default Histories

In [ ]:
# Load official SEC 10-K reported financials
raw_financials = load_sec_financials("../data/raw/edgar_sec_financials.csv")
defaults_df = load_corporate_defaults("../data/raw/corporate_defaults.csv")

merged_df = merge_financials_with_defaults(raw_financials, defaults_df)
print(f"Total SEC Filings Ingested: {len(merged_df)}")
print(f"Unique Corporate Counterparties: {merged_df['cik'].nunique()}")
print(f"Historical Default Events: {merged_df['target'].sum()}")
merged_df.head(5)


### 2. Preprocessing: Sector Median Imputation & Outlier Winsorization

In [ ]:
# Execute FR-1.3: Sector-specific median imputation and 1st/99th percentile winsorization
clean_df = preprocess_financial_data(merged_df, winsorize=True)
clean_df.describe().round(2)


### 3. Feature Engineering: Solvency, Leverage & Altman Z-Score

In [ ]:
# Compute fundamental ratios (FR-2)
features_df = compute_financial_ratios(clean_df)

ratio_cols = ['icr', 'debt_to_ebitda', 'current_ratio', 'ocf_to_debt', 'operating_margin', 'altman_z']
features_df[['company_name', 'sector', 'fiscal_year'] + ratio_cols].head(10)


### 4. Sector-Level Ratio Distribution Analysis

In [ ]:
# Compare Debt-to-EBITDA and Altman Z by Sector
summary_by_sector = features_df.groupby('sector')[ratio_cols].median()
summary_by_sector.round(3)
